In [21]:
import os
import pandas as pd


data_dir = '/home/volga/dev/python/ML/plastic_mapping/data'

training_data = pd.read_csv(os.path.join(data_dir, 'training_data.csv'))
testing_data = pd.read_csv(os.path.join(data_dir, 'testing_data.csv'))


In [ ]:

from sklearn.ensemble import RandomForestClassifier


def compute_indices(df):
    df = df.copy()
    avg_bg_nir = (df['blue_p50'] + df['green_p50'] + df['nir_p50']) / 3

    df['PMLI'] = (df['swir2_p50'] - df['red_p50']) / (df['swir2_p50'] + df['red_p50'])
    df['RPGI'] = df['blue_p50'] / (avg_bg_nir - 1) * 100
    df['NDBI'] = (df['swir1_p50'] - df['nir_p50']) / (df['swir1_p50'] + df['nir_p50'])
    df['VI'] = ((df['swir1_p50'] - df['nir_p50']) / (df['swir1_p50'] + df['nir_p50'])) * \
               ((df['nir_p50'] - df['red_p50']) / (df['nir_p50'] + df['red_p50']))
    df['PGI'] = df['blue_p50'] * (df['nir_p50'] - df['red_p50']) / (avg_bg_nir - 1) * 100
    df['NDVI'] = (df['nir_p50'] - df['red_p50']) / (df['nir_p50'] + df['red_p50'])
    return df

training_data = compute_indices(training_data)
testing_data = compute_indices(testing_data)


attributes = ['blue_p50', 'green_p50', 'nir_p50', 'nira_p50', 're1_p50', 're2_p50',
              're3_p50', 'red_p50', 'swir1_p50', 'swir2_p50', 'VV_p50', 'VH_p50',
              'PMLI', 'RPGI', 'NDBI', 'VI', 'PGI', 'NDVI']
target = 'TARGET'



clf = RandomForestClassifier(n_estimators=150, random_state=42)
clf.fit(training_data[attributes], training_data[target])

testing_data['classification'] = clf.predict(testing_data[attributes])

In [24]:
submission = testing_data[['ID']].copy()
submission['TARGET'] = clf.predict(testing_data[attributes])

submission.to_csv(os.path.join(data_dir, 'submission.csv'), index=False)